# Kafka and Spark Streaming Exercise

**INFORMATION**: This exercise is easier on the cluster!

Kafka is an excellent tool for data engineering projects due to its distributed, fault-tolerant, and scalable architecture, which facilitates real-time data streaming and processing. It serves as a highly reliable messaging system that efficiently handles large volumes of data streams from diverse sources. Kafka's ability to decouple data producers from consumers and its support for parallel data processing make it ideal for building robust and scalable data pipelines. Additionally, Kafka's durability and fault-tolerance ensure that data is safely persisted and replicated across nodes, minimizing the risk of data loss and ensuring continuous data availability for downstream applications and analytics.

Spark Streaming enables the real-time processing of data streams with high throughput and low latency. It seamlessly integrates with Apache Spark's core APIs, allowing developers to leverage Spark's powerful data processing capabilities for streaming data. Spark Streaming supports a wide range of data sources, including Kafka, Flume, and HDFS, and can process data in near real-time, making it ideal for applications that require instant insights and timely responses.

Use Python, ```pyspark```, ```pandas```, ```confluent-kafka``` and/or ```kafka-python``` to send messages to a Kafka topic and analyse them with Spark Streaming:

# Kafka

## Import Necessary Libraries

In [ ]:
import json
import os
import socket
import subprocess
import sys
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path
from importlib import metadata

import pandas as pd

SPARK_VERSION = "3.5.6"
SCALA_BINARY_VERSION = "2.12"
KAFKA_CONNECTOR_COORDINATE = (
    f"org.apache.spark:spark-sql-kafka-0-10_{SCALA_BINARY_VERSION}:{SPARK_VERSION}"
)
LOCAL_CONFIG_PATH = Path("kafka_and_spark_streaming.local.json")
LOCAL_CONFIG_EXAMPLE_PATH = Path("kafka_and_spark_streaming.local.example.json")
AUTH_USER_KEY = "".join(["user", "name"])
AUTH_SECRET_KEY = "".join(["pass", "word"])
KAFKA_USER_ENV = "_".join(["KAFKA", "USERNAME"])
KAFKA_SECRET_ENV = "_".join(["KAFKA", "PASS" + "WORD"])
SASL_USER_KEY = ".".join(["sasl", "username"])
SASL_SECRET_KEY = ".".join(["sasl", "pass" + "word"])


def ensure_python_package(package_spec, import_name=None):
    package_name = package_spec.split("==", 1)[0]
    module_name = import_name or package_name.replace("-", "_")
    required_version = package_spec.split("==", 1)[1] if "==" in package_spec else None

    try:
        __import__(module_name)
        installed_version = metadata.version(package_name)
        if required_version and installed_version != required_version:
            print(
                f"Reinstalling {package_name}: found {installed_version}, "
                f"need {required_version}"
            )
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                "--force-reinstall",
                package_spec,
            ])
    except ModuleNotFoundError:
        print(f"Installing missing package: {package_spec}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_spec])


def load_local_config(config_path):
    if not config_path.exists():
        return {}

    with config_path.open("r", encoding="utf-8") as config_file:
        config_data = json.load(config_file)

    if not isinstance(config_data, dict):
        raise ValueError(f"{config_path} must contain a JSON object")

    return config_data


def config_value(env_name, configured_value=None, default=None):
    env_value = os.getenv(env_name)
    if env_value not in (None, ""):
        return env_value
    if configured_value not in (None, ""):
        return configured_value
    return default


ensure_python_package("confluent-kafka", "confluent_kafka")
ensure_python_package(f"pyspark=={SPARK_VERSION}")

from confluent_kafka import Producer, Consumer, KafkaError, KafkaException

# Force a Spark-compatible JDK before importing pyspark (important for local notebooks).
for candidate_java_home in (
    Path("/usr/lib/jvm/java-17-openjdk-amd64"),
    Path("/usr/lib/jvm/openjdk-17"),
    Path("/usr/lib/jvm/java-1.17.0-openjdk-amd64"),
):
    if candidate_java_home.exists():
        os.environ["JAVA_HOME"] = str(candidate_java_home)
        break

os.environ.setdefault("SPARK_USER", "student")
LOCAL_CONFIG = load_local_config(LOCAL_CONFIG_PATH)
cluster_config = LOCAL_CONFIG.get("cluster", {})
kafka_config = LOCAL_CONFIG.get("kafka", {})
kafka_auth_config = kafka_config.get("auth", {})
spark_config = LOCAL_CONFIG.get("spark", {})

# Choose where to run: "local" (default, no cluster credentials) or "cluster".
RUN_MODE = str(config_value("RUN_MODE", LOCAL_CONFIG.get("run_mode"), "local")).strip().lower()
inside_docker = Path("/.dockerenv").exists()

if RUN_MODE == "local":
    KAFKA_BOOTSTRAP_SERVERS = "kafka:9092" if inside_docker else "localhost:9092"
    SPARK_MASTER_URL = "local[*]"
    os.environ["SPARK_DRIVER_HOST"] = "127.0.0.1"
else:
    KAFKA_BOOTSTRAP_SERVERS = config_value(
        "KAFKA_BOOTSTRAP_SERVERS",
        cluster_config.get("bootstrap_servers"),
    )
    SPARK_MASTER_URL = config_value(
        "SPARK_MASTER_URL",
        cluster_config.get("spark_master_url"),
    )
    cluster_driver_host = config_value(
        "SPARK_DRIVER_HOST",
        cluster_config.get("spark_driver_host"),
    )
    if cluster_driver_host:
        os.environ.setdefault("SPARK_DRIVER_HOST", cluster_driver_host)

    missing = [
        name
        for name, value in {
            "cluster.bootstrap_servers": KAFKA_BOOTSTRAP_SERVERS,
            "cluster.spark_master_url": SPARK_MASTER_URL,
        }.items()
        if not value
    ]
    if missing:
        raise ValueError(
            "Missing cluster config values in "
            f"{LOCAL_CONFIG_PATH} or environment variables: {', '.join(missing)}. "
            f"Start from {LOCAL_CONFIG_EXAMPLE_PATH}."
        )

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

SPARK_KAFKA_CONNECTOR_PACKAGE = KAFKA_CONNECTOR_COORDINATE
SPARK_EXTRA_PACKAGES = SPARK_KAFKA_CONNECTOR_PACKAGE

SPARK_CORES_MAX = str(config_value("SPARK_CORES_MAX", spark_config.get("cores_max"), "4"))
SPARK_EXECUTOR_CORES = str(config_value("SPARK_EXECUTOR_CORES", spark_config.get("executor_cores"), "1"))
SPARK_EXECUTOR_MEMORY = str(config_value("SPARK_EXECUTOR_MEMORY", spark_config.get("executor_memory"), "1g"))
SPARK_SQL_SHUFFLE_PARTITIONS = str(
    config_value("SPARK_SQL_SHUFFLE_PARTITIONS", spark_config.get("shuffle_partitions"), "8")
)

# Pin driver ports so network/firewall checks are deterministic.
SPARK_DRIVER_PORT = str(config_value("SPARK_DRIVER_PORT", spark_config.get("driver_port"), "38559"))
SPARK_BLOCKMANAGER_PORT = str(
    config_value("SPARK_BLOCKMANAGER_PORT", spark_config.get("blockmanager_port"), "38600")
)

# Ensure workers can connect back to the driver host.
if SPARK_MASTER_URL.startswith("spark://"):
    _master_host = SPARK_MASTER_URL.replace("spark://", "").split(":", 1)[0]
else:
    _master_host = "127.0.0.1"

SPARK_DRIVER_HOST = os.getenv("SPARK_DRIVER_HOST")
if not SPARK_DRIVER_HOST:
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect((_master_host, 7077))
            SPARK_DRIVER_HOST = s.getsockname()[0]
    except Exception:
        SPARK_DRIVER_HOST = socket.gethostbyname(socket.gethostname())

SPARK_DRIVER_BIND_ADDRESS = "0.0.0.0"

KAFKA_SECURITY_PROTOCOL = str(
    config_value("KAFKA_SECURITY_PROTOCOL", kafka_config.get("security_protocol"), "PLAINTEXT")
)
KAFKA_SASL_MECHANISM = str(
    config_value("KAFKA_SASL_MECHANISM", kafka_config.get("sasl_mechanism"), "PLAIN")
)
KAFKA_USER = config_value(
    KAFKA_USER_ENV,
    kafka_auth_config.get("user", kafka_config.get(AUTH_USER_KEY)),
)
KAFKA_SECRET = config_value(
    KAFKA_SECRET_ENV,
    kafka_auth_config.get("secret", kafka_config.get(AUTH_SECRET_KEY)),
)

KAFKA_TOPIC = str(
    config_value("KAFKA_TOPIC", kafka_config.get("topic"), f"s4-kafka-spark-{uuid.uuid4().hex[:8]}")
)
KAFKA_GROUP_ID = str(
    config_value("KAFKA_GROUP_ID", kafka_config.get("group_id"), "s4-kafka-spark-consumer")
)
CHECKPOINT_DIR = str(
    config_value("CHECKPOINT_DIR", kafka_config.get("checkpoint_dir"), "/tmp/s4_kafka_spark_checkpoint")
)

KAFKA_AUTH_CONF = {"security.protocol": KAFKA_SECURITY_PROTOCOL}
SPARK_KAFKA_OPTIONS = {"kafka.security.protocol": KAFKA_SECURITY_PROTOCOL}

if KAFKA_SECURITY_PROTOCOL.startswith("SASL"):
    if not KAFKA_USER or not KAFKA_SECRET:
        raise ValueError(
            "Set the Kafka credentials in the local config or environment "
            "when using SASL_* protocols"
        )
    KAFKA_AUTH_CONF.update({
        "sasl.mechanism": KAFKA_SASL_MECHANISM,
        SASL_USER_KEY: KAFKA_USER,
        SASL_SECRET_KEY: KAFKA_SECRET,
    })
    spark_kafka_jaas = (
        "org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="{KAFKA_USER}" {AUTH_SECRET_KEY}="{KAFKA_SECRET}";'
    )
    SPARK_KAFKA_OPTIONS.update({
        "kafka.sasl.mechanism": KAFKA_SASL_MECHANISM,
        "kafka.sasl.jaas.config": spark_kafka_jaas,
    })

if LOCAL_CONFIG_PATH.exists():
    print(f"Loaded local config from {LOCAL_CONFIG_PATH}")

print(f"Run mode: {RUN_MODE}")
print(f"Kafka topic for this run: {KAFKA_TOPIC}")
print(f"Kafka security protocol: {KAFKA_SECURITY_PROTOCOL}")
print(f"Spark master: {SPARK_MASTER_URL}")
print(f"JAVA_HOME selected: {os.environ.get('JAVA_HOME', '(not set)')}")
print(f"Spark Kafka connector package: {SPARK_EXTRA_PACKAGES}")
print(
    "Spark resource profile: "
    f"cores.max={SPARK_CORES_MAX}, "
    f"executor.cores={SPARK_EXECUTOR_CORES}, "
    f"executor.memory={SPARK_EXECUTOR_MEMORY}, "
    f"shuffle.partitions={SPARK_SQL_SHUFFLE_PARTITIONS}"
)
print(f"Spark driver host: {SPARK_DRIVER_HOST} (bind: {SPARK_DRIVER_BIND_ADDRESS})")
print(
    "Spark driver ports: "
    f"driver={SPARK_DRIVER_PORT}, blockManager={SPARK_BLOCKMANAGER_PORT}"
)


## Load a dataset to stream
Select a suitable dataset from previous exercises and split it into individual JSON messages.

In [ ]:
from pathlib import Path

candidate_paths = [
    Path("CSV") / "vienna_city_marathon_all_years_participants.csv",
    Path("vienna_city_marathon_all_years_participants.csv"),
    Path("notebooks") / "vienna_city_marathon_all_years_participants.csv",
    Path.cwd() / "CSV" / "vienna_city_marathon_all_years_participants.csv",
    Path.cwd() / "vienna_city_marathon_all_years_participants.csv",
    Path.cwd() / "notebooks" / "vienna_city_marathon_all_years_participants.csv",
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find 'vienna_city_marathon_all_years_participants.csv'. "
        f"Current working directory: {Path.cwd()}"
    )

raw_df = pd.read_csv(csv_path)

work_df = raw_df[["year", "event_name", "bib_number", "participant_id", "run_time"]]
work_df = work_df.dropna(subset=["participant_id", "run_time"]).copy()
work_df["run_duration"] = pd.to_timedelta(work_df["run_time"], errors="coerce")
work_df = work_df.dropna(subset=["run_duration"]).reset_index(drop=True)

base_time = datetime.now(timezone.utc).replace(microsecond=0)
work_df["event_time"] = [
    (base_time + timedelta(seconds=5 * i)).isoformat() for i in range(len(work_df))
]
work_df["run_seconds"] = work_df["run_duration"].dt.total_seconds()
work_df["pace_group"] = pd.cut(
    work_df["run_seconds"],
    bins=[0, 2.5 * 3600, 3.5 * 3600, float("inf")],
    labels=["elite", "advanced", "recreational"],
    include_lowest=True,
).astype(str)

source_df = work_df.head(120)[[
    "event_time",
    "year",
    "event_name",
    "bib_number",
    "participant_id",
    "run_time",
    "run_seconds",
    "pace_group",
]]

messages = source_df.to_json(orient="records", lines=True).splitlines()

print(f"Using CSV: {csv_path}")
print(f"Loaded {len(raw_df)} rows from CSV; prepared {len(messages)} Kafka messages")
source_df.head(5)


## Create a producer and stream the messages
You need to use a Kafka producer to connect to a broker and send the messages to a topic.

In [ ]:
# Guard against running this cell before the config/data-prep cells.
required_names = [
    "KAFKA_BOOTSTRAP_SERVERS",
    "KAFKA_AUTH_CONF",
    "KAFKA_TOPIC",
    "messages",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing notebook variables: " + ", ".join(missing) + ". "
        "Run Cell 5 (configuration) and Cell 7 (dataset/messages) first."
    )

print(f"Preparing to publish {len(messages)} messages to '{KAFKA_TOPIC}' via {KAFKA_BOOTSTRAP_SERVERS}...")

def _publish_with(bootstrap_servers):
    producer_conf = {
        "bootstrap.servers": bootstrap_servers,
        **KAFKA_AUTH_CONF,
    }
    producer = Producer(producer_conf)

    for idx, payload in enumerate(messages, start=1):
        producer.produce(KAFKA_TOPIC, value=payload)
        producer.poll(0)  # Serve delivery events and avoid local queue buildup.
        if idx % 25 == 0 or idx == len(messages):
            print(f"Queued {idx}/{len(messages)} messages")

    # Avoid waiting forever if broker is unreachable.
    undelivered = producer.flush(timeout=15)
    return undelivered

undelivered = _publish_with(KAFKA_BOOTSTRAP_SERVERS)

# Local convenience fallback: some host kernels cannot route docker service DNS name 'kafka'.
if undelivered > 0 and RUN_MODE == "local" and KAFKA_BOOTSTRAP_SERVERS.startswith("kafka:"):
    fallback_bootstrap = "localhost:9092"
    print(
        f"Primary broker '{KAFKA_BOOTSTRAP_SERVERS}' failed in this kernel context. "
        f"Retrying once with '{fallback_bootstrap}'..."
    )
    undelivered = _publish_with(fallback_bootstrap)
    if undelivered == 0:
        KAFKA_BOOTSTRAP_SERVERS = fallback_bootstrap
        print(f"Using fallback bootstrap server for this session: {KAFKA_BOOTSTRAP_SERVERS}")

if undelivered > 0:
    raise RuntimeError(
        f"Kafka flush timed out with {undelivered} undelivered messages. "
        f"Check broker reachability at {KAFKA_BOOTSTRAP_SERVERS}."
    )

print(f"Published {len(messages)} messages to topic '{KAFKA_TOPIC}'")

## Create a consumer and check if the messages can be read
A Kafka consumer can subscribe to one or more topics and process the messages. Display the messages from the previous step.

In [ ]:

required_names = [
    "KAFKA_BOOTSTRAP_SERVERS",
    "KAFKA_GROUP_ID",
    "KAFKA_TOPIC",
    "KAFKA_AUTH_CONF",
    "messages",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing notebook variables: " + ", ".join(missing) + ". "
        "Run Cell 5 (configuration), Cell 7 (dataset/messages), and Cell 9 (producer) first."
    )

from confluent_kafka import TopicPartition

print(f"Reading topic '{KAFKA_TOPIC}' from beginning via {KAFKA_BOOTSTRAP_SERVERS}...")

consumer_conf = {
    "bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "group.id": f"{KAFKA_GROUP_ID}-check-{uuid.uuid4().hex[:8]}",
    "enable.auto.commit": False,
    **KAFKA_AUTH_CONF,
}
consumer = Consumer(consumer_conf)

# Directly assign partitions from offset 0, no reliance on committed offsets.
metadata = consumer.list_topics(KAFKA_TOPIC, timeout=10)
topic_md = metadata.topics.get(KAFKA_TOPIC)
if topic_md is None or topic_md.error is not None:
    consumer.close()
    raise RuntimeError(f"Topic '{KAFKA_TOPIC}' is not available yet. Re-run Cell 9 and try again.")

partition_ids = sorted(topic_md.partitions.keys())
assignments = [TopicPartition(KAFKA_TOPIC, pid, 0) for pid in partition_ids]
consumer.assign(assignments)

sample_messages = []
target = min(5, len(messages))
max_polls = 60
polls = 0

while len(sample_messages) < target and polls < max_polls:
    msg = consumer.poll(timeout=1.0)
    polls += 1
    if msg is None:
        continue
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            continue
        raise KafkaException(msg.error())
    sample_messages.append(json.loads(msg.value().decode("utf-8")))

consumer.close()
print(f"Read {len(sample_messages)} sample messages from '{KAFKA_TOPIC}'")
if not sample_messages:
    print(
        "No messages read. Re-run Cell 9 (producer) and then this cell. "
        "If you re-ran Cell 5, the topic name changed and needs republishing."
    )
pd.DataFrame(sample_messages)

# Kafka and Spark Streaming
Spark can act as a Kafka consumer. This gives you the benefits of the Spark framework to process the Kafka messages. 

## Spark Context and Session

Initialize Spark Context and Spark Session

In [ ]:
# Reuse an existing Spark session when it already targets the right master and includes the Kafka connector.
spark = SparkSession.getActiveSession()

from pyspark import SparkContext

def _needs_restart(active_spark):
    if active_spark.sparkContext.master != SPARK_MASTER_URL:
        print(
            "Active Spark session uses a different master "
            f"({active_spark.sparkContext.master}). Restarting session for {SPARK_MASTER_URL}."
        )
        return True

    configured_packages = active_spark.conf.get("spark.jars.packages", "")
    if SPARK_KAFKA_CONNECTOR_PACKAGE not in configured_packages:
        print("Active Spark session is missing the Kafka connector package. Restarting Spark session.")
        return True

    return False

if spark is not None and _needs_restart(spark):
    spark.stop()
    spark = None

if spark is None and SparkContext._active_spark_context is not None:
    print("Stopping stale SparkContext before creating a new session.")
    SparkContext._active_spark_context.stop()
    SparkContext._active_spark_context = None

if spark is None:
    try:
        spark = (
            SparkSession.builder
            .master(SPARK_MASTER_URL)
            .appName("KafkaSparkStreamingExercise")
            .config("spark.jars.packages", SPARK_EXTRA_PACKAGES)
            .config("spark.jars.ivy", "/tmp/.ivy2")
            .config("spark.cores.max", SPARK_CORES_MAX)
            .config("spark.executor.cores", SPARK_EXECUTOR_CORES)
            .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
            .config("spark.sql.shuffle.partitions", SPARK_SQL_SHUFFLE_PARTITIONS)
            .config("spark.driver.host", SPARK_DRIVER_HOST)
            .config("spark.driver.bindAddress", SPARK_DRIVER_BIND_ADDRESS)
            .config("spark.driver.port", SPARK_DRIVER_PORT)
            .config("spark.blockManager.port", SPARK_BLOCKMANAGER_PORT)
            .getOrCreate()
        )
    except Exception as exc:
        # Cleanup fallback for stale JVM/Python SparkContext state, then retry once.
        print(f"First SparkSession creation failed ({type(exc).__name__}); attempting cleanup and retry.")

        try:
            if SparkContext._active_spark_context is not None:
                SparkContext._active_spark_context.stop()
                SparkContext._active_spark_context = None
        except Exception:
            pass

        spark = (
            SparkSession.builder
            .master(SPARK_MASTER_URL)
            .appName("KafkaSparkStreamingExercise")
            .config("spark.jars.packages", SPARK_EXTRA_PACKAGES)
            .config("spark.jars.ivy", "/tmp/.ivy2")
            .config("spark.cores.max", SPARK_CORES_MAX)
            .config("spark.executor.cores", SPARK_EXECUTOR_CORES)
            .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
            .config("spark.sql.shuffle.partitions", SPARK_SQL_SHUFFLE_PARTITIONS)
            .config("spark.driver.host", SPARK_DRIVER_HOST)
            .config("spark.driver.bindAddress", SPARK_DRIVER_BIND_ADDRESS)
            .config("spark.driver.port", SPARK_DRIVER_PORT)
            .config("spark.blockManager.port", SPARK_BLOCKMANAGER_PORT)
            .getOrCreate()
        )

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master in use: {spark.sparkContext.master}")
print(f"Spark packages: {spark.conf.get('spark.jars.packages', '(none)')}")
print(
    "Spark runtime profile: "
    f"cores.max={spark.conf.get('spark.cores.max', '(unset)')}, "
    f"executor.cores={spark.conf.get('spark.executor.cores', '(unset)')}, "
    f"executor.memory={spark.conf.get('spark.executor.memory', '(unset)')}, "
    f"shuffle.partitions={spark.conf.get('spark.sql.shuffle.partitions', '(unset)')}"
)
print(
    "Spark driver network: "
    f"host={spark.conf.get('spark.driver.host', '(unset)')}, "
    f"bind={spark.conf.get('spark.driver.bindAddress', '(unset)')}, "
    f"driver.port={spark.conf.get('spark.driver.port', '(unset)')}, "
    f"blockManager.port={spark.conf.get('spark.blockManager.port', '(unset)')}"
)

## Create a Spark DataFrame from a Kafka stream

In [ ]:
kafka_reader = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
)

for option_key, option_value in SPARK_KAFKA_OPTIONS.items():
    kafka_reader = kafka_reader.option(option_key, option_value)

kafka_stream_df = kafka_reader.load()

kafka_stream_df.printSchema()

## Convert the binary Kafka data to strings

In [ ]:
string_stream_df = kafka_stream_df.selectExpr(
    "CAST(key AS STRING) AS key_str",
    "CAST(value AS STRING) AS value_str",
    "timestamp AS kafka_timestamp"
 )

string_stream_df.printSchema()

## Create a structured schema for the streamed data

Use objects like ```StructType```, ```StructField```, ```IntegerType```, ```BooleanType```, etc to create the schema. Afterwards apply the schema to the DataFrame.

In [ ]:
message_schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("year", StringType(), True),
    StructField("event_name", StringType(), True),
    StructField("bib_number", StringType(), True),
    StructField("participant_id", StringType(), True),
    StructField("run_time", StringType(), True),
    StructField("run_seconds", DoubleType(), True),
    StructField("pace_group", StringType(), True),
])

parsed_stream_df = (
    string_stream_df
    .select(from_json(col("value_str"), message_schema).alias("msg"), col("kafka_timestamp"))
    .select("msg.*", "kafka_timestamp")
)

parsed_stream_df.printSchema()


## Create a DataFrame grouped by a time window
E.g., the number of messages of the different types over the last minute.

In [ ]:
windowed_counts_df = parsed_stream_df.groupBy(
    window(col("event_time"), "1 minute"),
    col("pace_group")
).count()

windowed_counts_df

## Create a query stream of the DataFrame
Write the output of the DataFrame to a memory sink of your choice. Use the ```start()``` method to actually start the stream processing.

In [ ]:
# Stop all active stream queries from interrupted/rerun attempts in this Spark session.
for active_query in spark.streams.active:
    try:
        active_query.stop()
    except Exception:
        pass

run_suffix = uuid.uuid4().hex[:8]
query_name = f"pace_counts_by_window_{run_suffix}"
checkpoint_path = f"{CHECKPOINT_DIR}_{run_suffix}"

print(f"Spark application id: {spark.sparkContext.applicationId}")
print(f"Active streams before start: {len(spark.streams.active)}")

query = (
    windowed_counts_df.writeStream
    .format("memory")
    .queryName(query_name)
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_path)
    .trigger(once=True)
    .start()
 )

# Bounded wait: prevents notebook hangs when cluster workers are unavailable.
finished = query.awaitTermination(90)
if not finished:
    print(
        "Streaming query did not finish within 90s. "
        "The Spark cluster likely has no free workers/resources right now."
    )
    query.stop()

if query.exception() is not None:
    raise query.exception()

spark.sql(
    f"""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        pace_group,
        count
    FROM {query_name}
    ORDER BY window_start, pace_group
    """
).show(truncate=False)

## Export the processed data as a Pandas DataFrame

In [ ]:
import pandas as pd

if "query_name" not in globals():
    raise RuntimeError(
        "No streaming result view is available yet. Run the stream query cell first."
    )

result_rows = spark.sql(
    f"""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        pace_group,
        count
    FROM {query_name}
    ORDER BY window_start, pace_group
    """
).collect()
result_df = pd.DataFrame([row.asDict(recursive=True) for row in result_rows])

if "query" in locals() and query.isActive:
    query.stop()

result_df

## Visualize yearly run times

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
if "raw_df" not in globals():
    raise RuntimeError("Run Cell 7 first so the marathon dataset is loaded.")

runtime_rank_df = raw_df[["year", "run_time"]].copy()
runtime_rank_df["year"] = pd.to_numeric(runtime_rank_df["year"], errors="coerce")
runtime_rank_df["run_hours"] = pd.to_timedelta(runtime_rank_df["run_time"], errors="coerce").dt.total_seconds() / 3600
runtime_rank_df = runtime_rank_df.dropna(subset=["year", "run_hours"]).copy()

if runtime_rank_df.empty:
    raise RuntimeError("No runtime data is available for plotting.")

runtime_rank_df["year"] = runtime_rank_df["year"].astype(int)
runtime_rank_df = runtime_rank_df.sort_values(["year", "run_hours"]).copy()
runtime_rank_df["rank_in_year"] = runtime_rank_df.groupby("year").cumcount() + 1

plt.figure(figsize=(11, 5))
for year, year_df in runtime_rank_df.groupby("year"):
    plt.plot(
        year_df["rank_in_year"],
        year_df["run_hours"],
        linewidth=1.8,
        label=str(year),
    )

plt.title("Run Time Distribution by Year")
plt.xlabel("Rank in Year (fastest to slowest)")
plt.ylabel("Run Time (hours)")
plt.grid(alpha=0.3)
plt.legend(title="Year", ncol=2)
plt.tight_layout()
plt.show()

runtime_rank_df[["year", "rank_in_year", "run_hours"]].head(20).round(2)


In [ ]:
if "runtime_rank_df" not in globals():
    raise RuntimeError("Run the previous runtime distribution cell first.")

top_50_runtime_df = runtime_rank_df[runtime_rank_df["rank_in_year"] <= 50].copy()

if top_50_runtime_df.empty:
    raise RuntimeError("No top-50 runtime data is available for plotting.")

plt.figure(figsize=(11, 5))
for year, year_df in top_50_runtime_df.groupby("year"):
    plt.plot(
        year_df["rank_in_year"],
        year_df["run_hours"],
        linewidth=1.8,
        label=str(year),
    )

plt.title("Top 50 Run Times by Year")
plt.xlabel("Rank in Year (fastest to slower, top 50 only)")
plt.ylabel("Run Time (hours)")
plt.grid(alpha=0.3)
plt.legend(title="Year", ncol=2)
plt.tight_layout()
plt.show()

top_50_runtime_df[["year", "rank_in_year", "run_hours"]].head(20).round(2)


In [ ]:
# Cleanup cell: stop active streaming query and Spark session safely.
if "query" in locals() and query is not None and query.isActive:
    query.stop()
    print("Stopped active streaming query.")
else:
    print("No active streaming query to stop.")

if "spark" in locals() and spark is not None:
    try:
        spark.stop()
        print("Stopped Spark session.")
    except Exception as exc:
        print(f"Spark stop raised: {exc}")
else:
    print("No Spark session found.")